# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [2]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [3]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad

    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    # TODO: implement
    x = x.detach().requires_grad_(True)
    f = (x ** 2).sum()
    grad = torch.autograd.grad(f, x)[0]
    return grad

In [4]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().

    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    # TODO: implement
    x = x.detach().requires_grad_(True)
    f = (x ** 2).sum()
    f.backward()
    return x.grad

In [5]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    # TODO: implement
    a = a.detach().requires_grad_(True)
    b = b.detach().requires_grad_(True)
    f = (a ** 2 + a * b).sum()
    grads = torch.autograd.grad(f, [a, b])
    return grads[0], grads[1]

## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [6]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        # TODO: implement
        self.x = x
        self.y = y

    def __len__(self) -> int:
        # TODO: implement
        return len(self.x)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.x[idx], self.y[idx]

In [8]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        # TODO: implement
        self.input_ids = tokens[:, :-1]
        self.target_ids = tokens[:, 1:]

    def __len__(self) -> int:
        # TODO: implement
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO: implement
        return self.input_ids[idx], self.target_ids[idx]

In [9]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        # TODO: implement
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)

    def __len__(self) -> int:
        # TODO: implement
        return len(self.tokens)

    def __getitem__(self, idx: int) -> torch.Tensor:
        # TODO: implement
        T_total = self.tokens.shape[1]
        max_start = T_total - self.crop_len
        s = torch.randint(0, max_start + 1, (1,), generator=self.generator).item()
        return self.tokens[idx, s:s + self.crop_len]

In [10]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    # TODO: implement
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    T_max = lengths.max().item()
    B = len(seqs)
    tokens = torch.full((B, T_max), pad_value, dtype=torch.long)
    for i, s in enumerate(seqs):
        tokens[i, :len(s)] = s.long()
    padding_mask = torch.arange(T_max).unsqueeze(0) >= lengths.unsqueeze(1)
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask)

In [11]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    # TODO: implement
    input_seqs = [item[0] for item in batch]
    target_seqs = [item[1] for item in batch]
    padded_inputs = pad_1d_sequences(input_seqs, pad_value)
    padded_targets = pad_1d_sequences(target_seqs, pad_value)
    return {
        "input_ids": padded_inputs.tokens,
        "target_ids": padded_targets.tokens,
        "attention_mask": ~padded_inputs.padding_mask,
        "padding_mask": padded_inputs.padding_mask,
    }

In [12]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    # TODO: implement
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )

## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [13]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    # TODO: implement
    with torch.no_grad():
        m = torch.zeros_like(p)
        v = torch.zeros_like(p)
    return AdamWState(m=m, v=v, t=0)

In [14]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    # TODO: implement
    with torch.no_grad():
        beta1, beta2 = betas
        state.t += 1

        state.m = beta1 * state.m + (1 - beta1) * grad
        state.v = beta2 * state.v + (1 - beta2) * grad ** 2

        m_hat = state.m / (1 - beta1 ** state.t)
        v_hat = state.v / (1 - beta2 ** state.t)

        p -= lr * (m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p)

    return state

In [15]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    # TODO: implement
    for i in range(len(params)):
        states[i] = adamw_step_(params[i], grads[i], states[i], lr, betas, eps, weight_decay)
    return states

## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [16]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    # TODO: implement
    model.train()
    optimizer.zero_grad()
    x, y = batch
    pred = model(x)
    loss = nn.functional.mse_loss(pred, y)
    loss.backward()
    optimizer.step()
    return loss.item()


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [17]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    # TODO: implement
    # For nn.Linear, weight shape is (out_features, in_features)
    fan_in = weight.shape[1]
    fan_out = weight.shape[0]
    return fan_in, fan_out

In [18]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    # TODO: implement
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * (6.0 / (fan_in + fan_out)) ** 0.5
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight

In [19]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    # TODO: implement
    fan_in, fan_out = fan_in_fan_out(weight)
    gain = 2.0 ** 0.5  # sqrt(2) for ReLU
    std = gain / fan_in ** 0.5
    bound = 3.0 ** 0.5 * std
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight

In [20]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    # TODO: implement
    with torch.no_grad():
        if scheme == "xavier":
            xavier_uniform_(layer.weight)
        elif scheme == "kaiming_relu":
            kaiming_uniform_(layer.weight, nonlinearity="relu")
        elif scheme == "zero":
            layer.weight.zero_()

        if layer.bias is not None:
            if scheme == "zero":
                layer.bias.zero_()
            else:
                layer.bias.zero_()
    return layer

## Testing

In [21]:
# ── Test Autograd ──
print("=== Testing Autograd ===")

# Test grad_with_autograd_grad
x = torch.tensor([1.0, 2.0, 3.0])
g = grad_with_autograd_grad(x)
expected = 2 * x  # df/dx = 2x for f(x) = sum(x^2)
print(f"grad_with_autograd_grad: shape={g.shape}, dtype={g.dtype}")
assert torch.allclose(g, expected), f"Expected {expected}, got {g}"

# Test grad_with_backward
g2 = grad_with_backward(x)
print(f"grad_with_backward: shape={g2.shape}, dtype={g2.dtype}")
assert torch.allclose(g2, expected), f"Expected {expected}, got {g2}"

# Test no gradient accumulation across calls
g2a = grad_with_backward(x)
g2b = grad_with_backward(x)
assert torch.allclose(g2a, g2b), "Gradients should not accumulate across calls"

# Test grad_wrt_multiple_inputs: f(a,b) = sum(a^2 + ab) -> df/da = 2a+b, df/db = a
a = torch.tensor([1.0, 2.0])
b = torch.tensor([3.0, 4.0])
da, db = grad_wrt_multiple_inputs(a, b)
expected_da = 2 * a + b
expected_db = a
print(f"grad_wrt_multiple_inputs: da.shape={da.shape}, db.shape={db.shape}")
assert torch.allclose(da, expected_da), f"df/da: expected {expected_da}, got {da}"
assert torch.allclose(db, expected_db), f"df/db: expected {expected_db}, got {db}"

print("All autograd tests passed!")

=== Testing Autograd ===
grad_with_autograd_grad: shape=torch.Size([3]), dtype=torch.float32
grad_with_backward: shape=torch.Size([3]), dtype=torch.float32
grad_wrt_multiple_inputs: da.shape=torch.Size([2]), db.shape=torch.Size([2])
All autograd tests passed!


In [22]:
# ── Test Datasets ──
print("=== Testing Datasets ===")

# Test TensorPairDataset
x = torch.randn(10, 4)
y = torch.randn(10, 1)
ds = TensorPairDataset(x, y)
assert len(ds) == 10, f"Expected len 10, got {len(ds)}"
xi, yi = ds[0]
print(f"TensorPairDataset: xi.shape={xi.shape}, yi.shape={yi.shape}, dtype={xi.dtype}")
assert torch.equal(xi, x[0]), "First item x should match"
assert torch.equal(yi, y[0]), "First item y should match"

# Test NextTokenDataset
tokens = torch.randint(0, 50, (5, 8))  # 5 sequences, length 8
nds = NextTokenDataset(tokens)
assert len(nds) == 5, f"Expected len 5, got {len(nds)}"
inp, tgt = nds[0]
print(f"NextTokenDataset: inp.shape={inp.shape}, tgt.shape={tgt.shape}, dtype={inp.dtype}")
assert inp.shape == (7,), f"Expected shape (7,), got {inp.shape}"
assert tgt.shape == (7,), f"Expected shape (7,), got {tgt.shape}"
assert torch.equal(inp, tokens[0, :-1]), "input_ids should be tokens[:, :-1]"
assert torch.equal(tgt, tokens[0, 1:]), "target_ids should be tokens[:, 1:]"
assert not inp.is_floating_point(), "dtype should remain integer"

# Test RandomCropSequenceDataset
tokens2 = torch.arange(100).reshape(5, 20)  # 5 sequences, length 20
crop_ds = RandomCropSequenceDataset(tokens2, crop_len=8, seed=42)
assert len(crop_ds) == 5
crop = crop_ds[0]
print(f"RandomCropSequenceDataset: crop.shape={crop.shape}, dtype={crop.dtype}")
assert crop.shape == (8,), f"Expected crop length 8, got {crop.shape}"

# Test determinism with same seed
crop_ds_a = RandomCropSequenceDataset(tokens2, crop_len=8, seed=42)
crop_ds_b = RandomCropSequenceDataset(tokens2, crop_len=8, seed=42)
assert torch.equal(crop_ds_a[0], crop_ds_b[0]), "Same seed should give same crop"

print("All dataset tests passed!")

=== Testing Datasets ===
TensorPairDataset: xi.shape=torch.Size([4]), yi.shape=torch.Size([1]), dtype=torch.float32
NextTokenDataset: inp.shape=torch.Size([7]), tgt.shape=torch.Size([7]), dtype=torch.int64
RandomCropSequenceDataset: crop.shape=torch.Size([8]), dtype=torch.int64
All dataset tests passed!


In [23]:
# ── Test Padding, Collation & DataLoader ──
print("=== Testing Padding & Collation ===")

# Test pad_1d_sequences
seqs = [torch.tensor([1, 2, 3]), torch.tensor([4, 5]), torch.tensor([6])]
padded = pad_1d_sequences(seqs, pad_value=0)
print(f"pad_1d_sequences: tokens.shape={padded.tokens.shape}, dtype={padded.tokens.dtype}, device={padded.tokens.device}")
print(f"  lengths={padded.lengths}, padding_mask=\n{padded.padding_mask}")
assert padded.tokens.shape == (3, 3), f"Expected (3,3), got {padded.tokens.shape}"
assert padded.tokens.dtype == torch.long
assert torch.equal(padded.lengths, torch.tensor([3, 2, 1]))
# Row 0: no padding
assert padded.padding_mask[0].sum() == 0
# Row 1: last position is padding
assert padded.padding_mask[1, 2] == True
assert padded.padding_mask[1, 1] == False
# Row 2: positions 1,2 are padding
assert padded.padding_mask[2, 0] == False
assert padded.padding_mask[2, 1] == True
assert padded.padding_mask[2, 2] == True
# Check actual token values
assert torch.equal(padded.tokens[0], torch.tensor([1, 2, 3]))
assert torch.equal(padded.tokens[1], torch.tensor([4, 5, 0]))
assert torch.equal(padded.tokens[2], torch.tensor([6, 0, 0]))

# Test collate_next_token_batch
batch = [
    (torch.tensor([1, 2, 3]), torch.tensor([2, 3, 4])),
    (torch.tensor([5, 6]), torch.tensor([6, 7])),
]
collated = collate_next_token_batch(batch, pad_value=0)
print(f"collate_next_token_batch keys: {list(collated.keys())}")
print(f"  input_ids.shape={collated['input_ids'].shape}, padding_mask.shape={collated['padding_mask'].shape}")
assert set(collated.keys()) == {"input_ids", "target_ids", "attention_mask", "padding_mask"}
assert collated["input_ids"].shape == (2, 3)
assert collated["target_ids"].shape == (2, 3)
# attention_mask should be the logical NOT of padding_mask
assert torch.equal(collated["attention_mask"], ~collated["padding_mask"])

# Test make_dataloader
simple_ds = TensorPairDataset(torch.randn(10, 3), torch.randn(10, 1))
dl = make_dataloader(simple_ds, batch_size=4, shuffle=False)
batch_x, batch_y = next(iter(dl))
print(f"DataLoader batch: x.shape={batch_x.shape}, y.shape={batch_y.shape}")
assert batch_x.shape == (4, 3)
assert batch_y.shape == (4, 1)

print("All padding/collation/dataloader tests passed!")

=== Testing Padding & Collation ===
pad_1d_sequences: tokens.shape=torch.Size([3, 3]), dtype=torch.int64, device=cpu
  lengths=tensor([3, 2, 1]), padding_mask=
tensor([[False, False, False],
        [False, False,  True],
        [False,  True,  True]])
collate_next_token_batch keys: ['input_ids', 'target_ids', 'attention_mask', 'padding_mask']
  input_ids.shape=torch.Size([2, 3]), padding_mask.shape=torch.Size([2, 3])
DataLoader batch: x.shape=torch.Size([4, 3]), y.shape=torch.Size([4, 1])
All padding/collation/dataloader tests passed!


In [24]:
# ── Test AdamW ──
print("=== Testing AdamW ===")

# Test init_adamw_state
p = torch.randn(3, 4, dtype=torch.float32)
state = init_adamw_state(p)
print(f"init_adamw_state: m.shape={state.m.shape}, v.shape={state.v.shape}, t={state.t}")
print(f"  m.dtype={state.m.dtype}, m.device={state.m.device}")
assert state.m.shape == p.shape
assert state.v.shape == p.shape
assert state.t == 0
assert torch.equal(state.m, torch.zeros_like(p))
assert torch.equal(state.v, torch.zeros_like(p))
assert state.m.dtype == p.dtype
assert not state.m.requires_grad
assert not state.v.requires_grad

# Test adamw_step_ against PyTorch's AdamW
torch.manual_seed(0)
lr = 0.01
betas = (0.9, 0.999)
eps = 1e-8
wd = 0.01

# Our custom implementation
p_custom = torch.randn(5, requires_grad=False).clone()
grad = torch.randn(5)
state = init_adamw_state(p_custom)
p_custom_before = p_custom.clone()
state = adamw_step_(p_custom, grad, state, lr=lr, betas=betas, eps=eps, weight_decay=wd)

# PyTorch's AdamW for comparison
p_pytorch = p_custom_before.clone().requires_grad_(True)
optimizer = torch.optim.AdamW([p_pytorch], lr=lr, betas=betas, eps=eps, weight_decay=wd)
p_pytorch.grad = grad.clone()
optimizer.step()

print(f"Custom AdamW result:  {p_custom[:3]}")
print(f"PyTorch AdamW result: {p_pytorch.data[:3]}")
assert torch.allclose(p_custom, p_pytorch.data, atol=1e-6), \
    f"Custom AdamW doesn't match PyTorch AdamW!\nCustom: {p_custom}\nPyTorch: {p_pytorch.data}"

# Test adamw_step_many_
torch.manual_seed(1)
params = [torch.randn(3), torch.randn(2, 2)]
grads = [torch.randn_like(p) for p in params]
states = [init_adamw_state(p) for p in params]
params_before = [p.clone() for p in params]
states = adamw_step_many_(params, grads, states, lr=lr, betas=betas, eps=eps, weight_decay=wd)
# Verify params were updated (changed from before)
for i in range(len(params)):
    assert not torch.equal(params[i], params_before[i]), f"Param {i} was not updated"
    assert states[i].t == 1, f"State {i} step should be 1"

print("All AdamW tests passed!")

=== Testing AdamW ===
init_adamw_state: m.shape=torch.Size([3, 4]), v.shape=torch.Size([3, 4]), t=0
  m.dtype=torch.float32, m.device=cpu
Custom AdamW result:  tensor([ 1.5508, -0.3034, -2.1886])
PyTorch AdamW result: tensor([ 1.5508, -0.3034, -2.1886])
All AdamW tests passed!


In [25]:
# ── Test Training Step ──
print("=== Testing Training Step ===")

torch.manual_seed(42)
model = nn.Linear(4, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(8, 4)
y = torch.randn(8, 1)

# Save params before step
w_before = model.weight.data.clone()
b_before = model.bias.data.clone()

loss_val = train_step_mse(model, (x, y), optimizer)
print(f"train_step_mse: loss={loss_val}, type={type(loss_val)}")
assert isinstance(loss_val, float), "Loss should be a Python float"
assert loss_val > 0, "MSE loss should be positive"

# Check that parameters were updated
assert not torch.equal(model.weight.data, w_before), "Weights should have changed after step"
assert not torch.equal(model.bias.data, b_before), "Bias should have changed after step"

# Verify loss matches manual computation
model2 = nn.Linear(4, 1)
model2.load_state_dict(model.state_dict())  # copy current state
with torch.no_grad():
    pred = model2(x)
    manual_loss = nn.functional.mse_loss(pred, y).item()
# Run another step and check loss is returned correctly
loss_val2 = train_step_mse(model2, (x, y), torch.optim.SGD(model2.parameters(), lr=0.01))
print(f"  manual MSE check: returned_loss={loss_val2:.6f}, manual_loss={manual_loss:.6f}")
assert abs(loss_val2 - manual_loss) < 1e-5, "Returned loss should match manual MSE computation"

print("All training step tests passed!")

=== Testing Training Step ===
train_step_mse: loss=3.112053871154785, type=<class 'float'>
  manual MSE check: returned_loss=2.932318, manual_loss=2.932318
All training step tests passed!


In [26]:
# ── Test Initialization ──
print("=== Testing Initialization ===")

# Test fan_in_fan_out
w = torch.randn(8, 4)  # (out_features=8, in_features=4)
fi, fo = fan_in_fan_out(w)
print(f"fan_in_fan_out: fan_in={fi}, fan_out={fo}")
assert fi == 4 and fo == 8, f"Expected (4, 8), got ({fi}, {fo})"

# Test xavier_uniform_ and compare with PyTorch's nn.init
torch.manual_seed(99)
w1 = torch.empty(64, 32)
xavier_uniform_(w1, gain=1.0)
expected_bound = (6.0 / (32 + 64)) ** 0.5
print(f"xavier_uniform_: min={w1.min():.4f}, max={w1.max():.4f}, expected_bound=+/-{expected_bound:.4f}")
assert w1.min() >= -expected_bound - 1e-6, "Values should be within bound"
assert w1.max() <= expected_bound + 1e-6, "Values should be within bound"

# Compare with PyTorch built-in
torch.manual_seed(99)
w1_ref = torch.empty(64, 32)
nn.init.xavier_uniform_(w1_ref, gain=1.0)
assert torch.equal(w1, w1_ref), "xavier_uniform_ should match nn.init.xavier_uniform_"

# Test kaiming_uniform_ and compare with PyTorch's nn.init
torch.manual_seed(99)
w2 = torch.empty(64, 32)
kaiming_uniform_(w2, nonlinearity="relu")
gain = 2.0 ** 0.5
std = gain / 32 ** 0.5
expected_bound_k = 3.0 ** 0.5 * std
print(f"kaiming_uniform_: min={w2.min():.4f}, max={w2.max():.4f}, expected_bound=+/-{expected_bound_k:.4f}")
assert w2.min() >= -expected_bound_k - 1e-6, "Values should be within bound"
assert w2.max() <= expected_bound_k + 1e-6, "Values should be within bound"

# Compare with PyTorch built-in
torch.manual_seed(99)
w2_ref = torch.empty(64, 32)
nn.init.kaiming_uniform_(w2_ref, nonlinearity="relu")
assert torch.equal(w2, w2_ref), "kaiming_uniform_ should match nn.init.kaiming_uniform_"

# Test init_linear_ with "zero" scheme
layer_zero = nn.Linear(4, 3)
init_linear_(layer_zero, scheme="zero")
print(f"init_linear_ zero: weight sum={layer_zero.weight.sum():.1f}, bias sum={layer_zero.bias.sum():.1f}")
assert torch.equal(layer_zero.weight, torch.zeros(3, 4)), "Zero scheme: weights should be all zeros"
assert torch.equal(layer_zero.bias, torch.zeros(3)), "Zero scheme: bias should be all zeros"

# Test init_linear_ with "xavier"
layer_xavier = nn.Linear(4, 3)
init_linear_(layer_xavier, scheme="xavier")
assert not torch.equal(layer_xavier.weight, torch.zeros(3, 4)), "Xavier: weights should not be zeros"
assert torch.equal(layer_xavier.bias, torch.zeros(3)), "Xavier: bias should be zeros"
print(f"init_linear_ xavier: weight sample={layer_xavier.weight[0, :2]}, bias={layer_xavier.bias}")

# Test init_linear_ with "kaiming_relu"
layer_kaiming = nn.Linear(4, 3)
init_linear_(layer_kaiming, scheme="kaiming_relu")
assert not torch.equal(layer_kaiming.weight, torch.zeros(3, 4)), "Kaiming: weights should not be zeros"
assert torch.equal(layer_kaiming.bias, torch.zeros(3)), "Kaiming: bias should be zeros"
print(f"init_linear_ kaiming: weight sample={layer_kaiming.weight[0, :2]}, bias={layer_kaiming.bias}")

print("All initialization tests passed!")

=== Testing Initialization ===
fan_in_fan_out: fan_in=4, fan_out=8
xavier_uniform_: min=-0.2500, max=0.2496, expected_bound=+/-0.2500
kaiming_uniform_: min=-0.4330, max=0.4324, expected_bound=+/-0.4330
init_linear_ zero: weight sum=0.0, bias sum=0.0
init_linear_ xavier: weight sample=tensor([ 0.5568, -0.7540], grad_fn=<SliceBackward0>), bias=Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
init_linear_ kaiming: weight sample=tensor([-0.1623, -0.4021], grad_fn=<SliceBackward0>), bias=Parameter containing:
tensor([0., 0., 0.], requires_grad=True)
All initialization tests passed!
